# Generate molecules from saved GraphVAE model

In [1]:
import sys
from pathlib import Path

import torch
from rdkit import Chem
from rdkit.Chem import Draw
import numpy as np
sys.path.append(str(Path.cwd().resolve().parent))

from src.GNN_VAETorch import GraphVAE
import pandas as pd
from rdkit import RDLogger
from src.preprocessingTorch import MolDataModule

RDLogger.DisableLog("rdApp.*")


In [2]:
LATENT_DIM = 128
MAX_ATOM = 50
MODEL_PATH = Path('../model/vae_model.pt')
OUT_DIR = Path('../img/generated')
OUT_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = GraphVAE(latent_dim=LATENT_DIM).to(device)
state_dict = torch.load(MODEL_PATH, map_location=device)
model.load_state_dict(state_dict)
model.eval()

print(f'Loaded model from: {MODEL_PATH.resolve()}')
print(f'Using device: {device}')

Loaded model from: D:\project\python\ds\deep learning\proj\model\vae_model.pt
Using device: cpu


In [3]:


BOND_TYPES = [
    Chem.BondType.SINGLE,
    Chem.BondType.DOUBLE,
    Chem.BondType.TRIPLE,
    Chem.BondType.AROMATIC,
]

VALENCE_MAP = {
    1: 1,
    6: 4,
    7: 3,
    8: 2,
    9: 1,
    15: 5,
    16: 6,
    17: 1,
    35: 1,
    53: 1,
}


def get_bond_order(bond_type):
    if bond_type == Chem.BondType.SINGLE:
        return 1.0
    if bond_type == Chem.BondType.DOUBLE:
        return 2.0
    if bond_type == Chem.BondType.TRIPLE:
        return 3.0
    if bond_type == Chem.BondType.AROMATIC:
        return 1.5
    return 0.0


def nearest_allowed_atom(value, allowed_atoms):
    if not np.isfinite(value):
        return None
    v = int(round(float(value)))
    if v < 1 or v > 118:
        return None
    return int(min(allowed_atoms, key=lambda x: abs(x - v)))


def infer_allowed_atoms_from_data(csv_path, top_k=12):
    df = pd.read_csv(csv_path)
    counts = {}
    for s in df["smiles"].dropna():
        m = Chem.MolFromSmiles(str(s))
        if m is None:
            continue
        for a in m.GetAtoms():
            z = int(a.GetAtomicNum())
            counts[z] = counts.get(z, 0) + 1

    allowed = sorted(counts, key=counts.get, reverse=True)[:top_k]
    if not allowed:
        allowed = [6, 7, 8, 9, 15, 16, 17, 35, 53]
    return allowed


@torch.no_grad()
def build_latent_bank(model, csv_path, device, bank_size=4096, batch_size=128):
    dm = MolDataModule(csv_path=csv_path, batch_size=batch_size, test_size=0.2)
    train_loader, _ = dm.setup()

    chunks = []
    total = 0
    for batch in train_loader:
        batch = tuple(x.to(device) for x in batch)
        node_f, edge_idx, edge_f, adj, mask, global_f = batch
        z_mean, _ = model.encoder(node_f, edge_idx, edge_f, adj, mask, global_f)
        z_mean = z_mean.detach().cpu()
        chunks.append(z_mean)
        total += z_mean.shape[0]
        if total >= bank_size:
            break

    if not chunks:
        return None
    return torch.cat(chunks, dim=0)[:bank_size]


def decode_one(node_feat, adj_prob, edge_feat, allowed_atoms, adj_threshold=0.35):
    mol = Chem.RWMol()
    graph_to_mol = {}
    valence_used = {}
    valence_max = {}

    for i in range(node_feat.shape[0]):
        if float(np.max(adj_prob[i])) < adj_threshold:
            continue

        atomic_num = nearest_allowed_atom(node_feat[i, 0], allowed_atoms)
        if atomic_num is None:
            continue

        mol_idx = mol.AddAtom(Chem.Atom(int(atomic_num)))
        graph_to_mol[i] = mol_idx
        valence_used[mol_idx] = 0.0
        valence_max[mol_idx] = float(VALENCE_MAP.get(int(atomic_num), 4))

    if len(graph_to_mol) < 2:
        return None

    candidates = []
    graph_indices = sorted(graph_to_mol.keys())
    for ii, i in enumerate(graph_indices):
        for j in graph_indices[ii + 1 :]:
            p = float(adj_prob[i, j])
            if p < adj_threshold:
                continue

            bond_idx = int(np.argmax(edge_feat[i, j, :4]))
            bond_type = BOND_TYPES[bond_idx]
            if bond_type == Chem.BondType.AROMATIC:
                bond_type = Chem.BondType.SINGLE

            candidates.append((p, i, j, bond_type))

    candidates.sort(key=lambda x: x[0], reverse=True)

    for _, gi, gj, bond_type in candidates:
        a = graph_to_mol[gi]
        b = graph_to_mol[gj]

        if mol.GetBondBetweenAtoms(a, b) is not None:
            continue

        order = get_bond_order(bond_type)
        if order <= 0.0:
            continue

        if valence_used[a] + order > valence_max[a]:
            continue
        if valence_used[b] + order > valence_max[b]:
            continue

        mol.AddBond(a, b, bond_type)
        valence_used[a] += order
        valence_used[b] += order

    out = mol.GetMol()
    try:
        Chem.SanitizeMol(out)
    except Exception:
        return None

    if out.GetNumBonds() == 0:
        return None

    return out


@torch.no_grad()
def sample_one_molecule(
    model,
    latent_dim,
    max_atom,
    allowed_atoms,
    latent_bank=None,
    posterior_mix=0.85,
    posterior_noise=0.20,
    temperature=0.8,
    adj_threshold=0.35,
):
    use_posterior = latent_bank is not None and np.random.rand() < posterior_mix

    if use_posterior:
        idx = np.random.randint(0, latent_bank.shape[0])
        z = latent_bank[idx : idx + 1].to(device)
        z = z + torch.randn(1, latent_dim, device=device) * posterior_noise
    else:
        z = torch.randn(1, latent_dim, device=device) * temperature

    node_f, adj_prob, edge_f, _ = model.decoder(z)

    node_np = node_f[0].detach().cpu().numpy()
    adj_np = adj_prob[0].detach().cpu().numpy()
    edge_np = edge_f[0].detach().cpu().numpy()

    mol = decode_one(
        node_np,
        adj_np,
        edge_np,
        allowed_atoms=allowed_atoms,
        adj_threshold=adj_threshold,
    )
    if mol is None:
        return None

    smiles = Chem.MolToSmiles(mol, canonical=True)
    if not smiles:
        return None

    return mol, smiles


CSV_PATH = Path("../data/zinc.csv")
ALLOWED_ATOMS = infer_allowed_atoms_from_data(CSV_PATH, top_k=12)
latent_bank = build_latent_bank(
    model,
    csv_path=CSV_PATH,
    device=device,
    bank_size=4096,
    batch_size=128,
)

print("Allowed atoms (from dataset):", ALLOWED_ATOMS)
print("Latent bank size:", 0 if latent_bank is None else int(latent_bank.shape[0]))

num_samples = 4000
temperature = 0.8
adj_threshold = 0.35
valid = 0
seen = set()
results = []

for i in range(num_samples):
    out = sample_one_molecule(
        model,
        latent_dim=LATENT_DIM,
        max_atom=MAX_ATOM,
        allowed_atoms=ALLOWED_ATOMS,
        latent_bank=latent_bank,
        posterior_mix=0.85,
        posterior_noise=0.20,
        temperature=temperature,
        adj_threshold=adj_threshold,
    )
    if out is None:
        continue

    mol, smiles = out
    if smiles in seen:
        continue
    seen.add(smiles)

    img_path = OUT_DIR / f"mol_{i:04d}.png"
    Draw.MolToImage(mol, size=(512, 512), kekulize=True).save(str(img_path))
    results.append((str(img_path), smiles))
    valid += 1

print(f"Valid unique molecules: {valid}/{num_samples}")
for p, s in results[:50]:
    print(f"{p}  ->  {s}")


Allowed atoms (from dataset): [6, 7, 8, 16, 9, 17, 35, 53, 15]
Latent bank size: 4096
Valid unique molecules: 361/4000
..\img\generated\mol_0000.png  ->  CCCCCCCNC1NNNNNNNNNNN2NNNCCCCCCCCCCCCCC12
..\img\generated\mol_0001.png  ->  CNCCCNNNNNNN1NNNNNNNNNNNNN2NNNCCCCC3CCCCC321
..\img\generated\mol_0002.png  ->  CNCCNNNNNNNN1NNNNNNNNNNNNN2NNCCCCCC3CCCCC321
..\img\generated\mol_0003.png  ->  CCCCCCCNN1NNNNNNNNNNN2NNNNNCCCCCCCCCCCC12
..\img\generated\mol_0004.png  ->  CNCCNNNNN1NNNNNNNNNNNNNNNN2CCCCCCCC3CCCC1C32
..\img\generated\mol_0005.png  ->  CCCCNNNNNNNNNNNNNNNNNNNNN1NNCNCCCC2CCCCC21
..\img\generated\mol_0006.png  ->  CNCCNNNNNNNNNNNNNNNNNNNNN1CCCCCCCC2CCCCC21
..\img\generated\mol_0007.png  ->  CNCCNNNNN1CCCCCCCCCCCCNNNNNNNNNNNNNNNN1
..\img\generated\mol_0008.png  ->  CNNCNNNNNNNNNNNN1NNNNNNNNN2NNNNNNCC3CCCCC321
..\img\generated\mol_0009.png  ->  CNCCNNNNNNNNNNNNNNNNNNNNN1NNCCCCCC2CCCCC21
..\img\generated\mol_0010.png  ->  CNNCNNNNNNNNNNNN1NNNNNNNNN2NNNNNNCN3CCCCC321
..\img\generated\m